# FINAL — Task A (Hate / Non-Hate)

**Settings:** Accelerator = **GPU T4 ×2** · Internet = **On** · khoảng 3–3,5 giờ nếu chạy hết

## Dữ liệu
```
binary_train.csv ── làm sạch, bỏ trùng ──► 6.386 câu ─┬─► HELD-OUT 639 câu   (chấm điểm; KHÔNG BAO GIỜ đổi)
                                                      └─► FIT 5.747 câu
binary_validation_inputs.csv (806, không nhãn)                   │
   gán nhãn theo cấu trúc A–B: id có trong file B → Hate (395)   │
                               ngược lại          → Non-Hate (411)│
   bỏ 14 câu trùng train ──────────────── thêm 792 câu ───────────┘──► FIT 6.539 câu
hastika_binary_test.csv (806) ──► CHỈ để dự đoán. Nhãn test luôn do MODEL đoán.
```

## Model và ensemble
```
            ┌ ① cnerg_muril gốc              × 3 seed ─ TB ┐
câu ──┬──►  ├ ② cnerg_muril + MLM bản 2      × 3 seed ─ TB ┼─ TB ─► P_trans ─┐
      │     └ ③ cnerg_muril + MLM + vocab    × 3 seed ─ TB ┘                 ├─ W·P_trans + (1−W)·P_tfidf ─► nhãn
      └──►    ④ TF-IDF LR   ⑤ TF-IDF SVM   ⑥ TF-IDF SGD ──────── TB ─► P_tfidf ┘
```
- **Tầng 1:** trung bình 3 seed của mỗi transformer. **Tầng 2:** trung bình trong mỗi nhóm.
  **Tầng 3:** trộn hai nhóm với trọng số **cố định** `W` (không tối ưu trên held-out).
- **Bản 90%:** chấm điểm được trên held-out. **Bản 100% (`_full`):** gộp cả held-out vào fit,
  dùng đúng số epoch / C mà bản 90% đã chọn, nên không chấm điểm được.

> ⚠️ **Phải khai báo trong system paper:** nhãn của 806 câu val được **suy ra** từ việc id có mặt
> trong file Task B (gồm cả file B test), không phải nhãn do ban tổ chức công bố. Quy tắc này đúng
> 100% trên A train (3.160/3.160 Hate, 0/3.286 Non-Hate). File nộp **val** của notebook này **vô
> nghĩa để đánh giá** vì model đã học chính các câu đó. File nộp **test** mới là bài nộp.

In [ ]:
import os, subprocess, sys

REPO, BRANCH, WORK = "trong5nhan6/Text", "main", "/kaggle/working"
TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("GH_TOKEN")
except Exception:
    pass
url = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
hide = (lambda s: s.replace(TOKEN, "***")) if TOKEN else (lambda s: s)

os.chdir(WORK)
cmd = (["git", "-C", "repo", "pull", "--ff-only"] if os.path.isdir("repo/.git")
       else ["git", "clone", "--depth", "1", "-b", BRANCH, url, "repo"])
r = subprocess.run(cmd, capture_output=True, text=True)
print(hide((r.stdout + r.stderr).strip()))
if r.returncode:
    raise SystemExit("git that bai -- kiem tra Internet = On, repo Public")

os.chdir(f"{WORK}/repo"); sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
!pip -q install ftfy sentencepiece gdown
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ls data/raw

## 1) Bảng điều khiển

In [ ]:
# ============================== BANG DIEU KHIEN ==============================
SEEDS    = [42, 43, 44]
EPOCHS   = 10            # fine-tune; luu epoch tot nhat theo held-out
VAL_LEAK = False         # True: them 792 cau val (nhan suy ra tu file B) vao phan fit.
                         # PHAI khop voi RESULTS_ZIP: zip noleak <-> False, zip leak (_vl) <-> True
RUN_FULL = True          # them ban 100% (gop held-out vao fit) -- khong cham diem duoc

TRANSFORMERS = [   # (ten, config, model.name -- None = mac dinh cua config)
    ('cnerg',  'cnerg_muril', None),
    ('mlm_v2', 'cnerg_muril', 'checkpoints/mlm_v2/cnerg-muril'),
    ('mlm_vx', 'cnerg_muril', 'checkpoints/mlm_vx/cnerg-muril'),
]
TFIDF   = ['lr', 'svm', 'sgd']
W_TRANS = 0.6            # trong so nhom transformer; TF-IDF = 1 - W_TRANS

# checkpoint MLM: thu muc dich -> (ten file zip, Google Drive: DAN NGUYEN LINK chia se hoac chi file id)
# Link dang https://drive.google.com/file/d/<ID>/view?usp=sharing, che do "Anyone with the link".
MLM_CKPTS = {
    'checkpoints/mlm_v2/cnerg-muril': ('checkpoints_mlm_v2_cnerg-muril.zip', 'https://drive.google.com/file/d/1r5y4vY3jLBpvautjkq5djeJ7sI2ODRtS/view?usp=drive_link'),   # <- dan link Drive vao ''
    'checkpoints/mlm_vx/cnerg-muril': ('checkpoints_mlm_vx_cnerg-muril.zip', 'https://drive.google.com/file/d/1mGF5eb3B-DdwXoYJOSqPrZC5m4X4oiIe/view?usp=drive_link'),   # <- dan link Drive vao ''
}
SUFFIX = f'_e{EPOCHS}'

# ---- TANG 2: LLM ----------------------------------------------------------------
USE_LLM    = True            # them LLM va ghep 2 tang
LLM_DELTA  = 0.2             # cau co |P_ensemble(Hate) - 0,5| < LLM_DELTA = "vung khong chac" -> hoi LLM
LLM_MIX    = 0.5             # trong vung do: P = LLM_MIX * P_ensemble + (1 - LLM_MIX) * P_LLM
                             #   0 = LLM QUYET HOAN TOAN cac cau ensemble khong chac | 0.5 = trung binh | 1 = bo LLM
                             #   (bang muc 7 va file nop luon co them ban mix 0 de so sanh: a_test_final_llm_mix0)
LLM_MODE   = 'finetune'      # 'finetune' = train LoRA + head (~1 gio voi 7B)
                             # 'zeroshot' = KHONG train: dinh nghia nhan + binh luan -> so P("Hate") voi P("Non-Hate")
                             # 'fewshot'  = KHONG train: nhu zeroshot + LLM_K cau giong nhat trong phan fit kem nhan
LLM_K      = 8               # fewshot: so vi du moi cau (8-16; nhieu hon = prompt dai, cham hon)
LLM_SCOPE  = 'band'          # zeroshot/fewshot: cau nao dua cho LLM cham
                             #   'band' = CHI cau ensemble khong chac (|P - 0,5| < LLM_DELTA) -> nhanh ~4 lan, chi dung cho 2 tang
                             #   'all'  = MOI cau held-out/val/test -> them file nop a_test_final_llmonly (LLM tu quyet ca file test)
                             # (finetune luon du doan moi cau: phan du doan re so voi phan train)

# ---- cau hinh train LLM ------------------------------------------------------------
# Chon 1 PRESET theo suc GPU; moi preset da chon san model + 4-bit + batch phu hop 1 T4 (15 GB).
#   qwen7b   Qwen2.5-7B-Instruct   4-bit, batch 8x4, grad ckpt   ~40 phut/epoch (manh nhat)
#   qwen3b   Qwen2.5-3B-Instruct   fp16,  batch 8x4, grad ckpt   ~40 phut
#   qwen1.5b Qwen2.5-1.5B-Instruct fp16,  batch 16x2             ~20 phut
#   qwen0.5b Qwen2.5-0.5B-Instruct fp16,  batch 32x1             ~10 phut     (thu nhanh / CPU)
#   sarvam2b sarvamai/sarvam-1 (2B, chuyen tieng An Do) fp16, batch 8x4, grad ckpt
# Neu loss ra NaN voi fp16 (Qwen duoc train bf16, T4 khong co bf16): bat LLM_4BIT = True.
LLM_PRESET = 'qwen7b'
LLM_PRESETS = {
    'qwen7b':   dict(model='Qwen/Qwen2.5-7B-Instruct',   four_bit=True,  batch=8,  accum=4, ckpt=True,  lr=1e-4),
    'qwen3b':   dict(model='Qwen/Qwen2.5-3B-Instruct',   four_bit=False, batch=8,  accum=4, ckpt=True,  lr=1e-4),
    'qwen1.5b': dict(model='Qwen/Qwen2.5-1.5B-Instruct', four_bit=False, batch=16, accum=2, ckpt=False, lr=2e-4),
    'qwen0.5b': dict(model='Qwen/Qwen2.5-0.5B-Instruct', four_bit=False, batch=32, accum=1, ckpt=False, lr=2e-4),
    'sarvam2b': dict(model='sarvamai/sarvam-1',          four_bit=False, batch=8,  accum=4, ckpt=True,  lr=1e-4),
}
_p = LLM_PRESETS[LLM_PRESET]

# ---- tung tham so (mac dinh lay tu preset; sua truc tiep o day de ghi de) ------------
LLM_MODEL       = _p['model']     # hoac dan ten model HuggingFace bat ky (LLM giai ma, co q_proj/v_proj...)
LLM_4BIT        = _p['four_bit']  # nap trong so 4-bit (bat buoc voi 7B tren T4)
LLM_BATCH       = _p['batch']     # cau / buoc tren 1 GPU. OOM -> chia doi
LLM_GRAD_ACCUM  = _p['accum']     # cong don gradient: batch hieu dung = LLM_BATCH x LLM_GRAD_ACCUM
LLM_GRAD_CKPT   = _p['ckpt']      # tinh lai activation luc backward: it bo nho hon, cham hon ~30%
LLM_LR          = _p['lr']        # lr cua LoRA + head
LLM_LORA_R      = 16              # hang cua LoRA (8 / 16 / 32): lon hon = nhieu tham so train hon
LLM_LORA_ALPHA  = 32              # he so scale cua LoRA, thuong = 2 x LLM_LORA_R
LLM_LORA_DROP   = 0.05            # dropout trong LoRA
LLM_EPOCHS      = 3               # so epoch (cung la do dai lich LR); luu epoch tot nhat theo held-out.
                                  # 7B 4-bit tren T4 ~40 phut/epoch; lan chay 10 epoch dat dinh o epoch 3 roi overfit
LLM_PATIENCE    = 1               # dung som: dung sau LLM_PATIENCE epoch eval F1 khong tang (LLM_EPOCHS = tat)
LLM_MAX_LEN     = 192             # token = binh luan (<=96 cho 98% cau) + khung prompt (~45)
LLM_AUX_TARGET  = True            # head phu "doi tuong bi tan cong" (nhom Task B), chi dung khi train
LLM_TARGET_W    = 0.5             # trong so loss cua head phu
LLM_SEED        = 42
LLM_PROMPT = (                    # khung bao moi binh luan; {text} = binh luan. Giong het luc train va du doan.
    'Below is a YouTube comment written in Kannada-English code-mixed text (Kannada in Latin script).\n'
    'Comment: "{text}"\n'
    'Question: who is targeted by this comment, and is it hate speech?'
)
# Ket qua cu (tranh train lai): zip results_final_*.zip tu lan chay truoc. Mot trong ba:
#   link Google Drive (Anyone with the link) | duong dan file trong /kaggle/input | '' = tu tim
#   moi results*.zip trong /kaggle/input. Link duoi day = results_final_noleak.zip (24 run, VAL_LEAK=False).
RESULTS_ZIP = 'https://drive.google.com/file/d/1j2Ga6AJ7KU18yk_d-QH8c4GaY2qtVHDO/view?usp=drive_link'
# ===================================================================================

## 2) Lấy checkpoint MLM

Hai checkpoint (mỗi cái khoảng 0,9 GB): `mlm_v2/cnerg-muril` (MLM bản 2) và `mlm_vx/cnerg-muril`
(MLM + mở rộng vocab). Cell dưới thử lần lượt:

1. **Kaggle Dataset đã giải nén** (*Add Input*): tìm `…/mlm_v2/cnerg-muril/config.json` ở **mọi độ
   sâu** trong `/kaggle/input`, rồi trỏ tới (symlink, không chép).
2. **File zip** trong `/kaggle/input`, hoặc **tải từ Google Drive** (dán link vào `MLM_CKPTS`).

File zip có thể có **lớp thư mục bọc ngoài** hoặc không, ví dụ
`checkpoints_mlm_v2_cnerg-muril/checkpoints/mlm_v2/cnerg-muril/…` (nén lại sau khi giải nén trên
Windows) hoặc `checkpoints/mlm_v2/cnerg-muril/…` (zip gốc từ Kaggle). Cell giải nén vào thư mục
tạm, **tìm** đúng thư mục chứa `config.json`, rồi chuyển về `checkpoints/mlm_v2/cnerg-muril`. Cả hai
dạng đều chạy được.

In [ ]:
import glob, zipfile, shutil, json

def find_ckpt(root, tail):
    """thu muc chua config.json co duong dan ket thuc bang `tail`, o bat ky do sau nao"""
    hits = sorted(glob.glob(f"{root}/**/{tail}/config.json", recursive=True), key=len)
    return os.path.dirname(hits[0]) if hits else None

def drive_download(link, out):
    """link chia se Drive (.../file/d/<ID>/view?...) hoac chi <ID>. Tu tach ID roi goi
    gdown.download(id=...), cach moi phien ban gdown deu ho tro (gdown 6 da bo `fuzzy`)."""
    import gdown, re
    m = re.search(r"/d/([\w-]+)", link) or re.search(r"[?&]id=([\w-]+)", link)
    gdown.download(id=m.group(1) if m else link.strip(), output=out, quiet=False)
    if not zipfile.is_zipfile(out):
        raise SystemExit(f"{out} khong phai file zip -- kiem tra link Drive da de 'Anyone with the link' chua")

for d, (zname, drive) in MLM_CKPTS.items():
    if os.path.isfile(f"{d}/config.json"):
        print("da co:", d); continue
    tail = d.split("checkpoints/", 1)[1]                       # vd mlm_v2/cnerg-muril
    src = find_ckpt("/kaggle/input", tail)
    if src:                                                    # 1) dataset da giai nen
        os.makedirs(os.path.dirname(d), exist_ok=True)
        os.symlink(src, d)
        print(f"{d} -> {src} (symlink)"); continue
    zips = glob.glob(f"/kaggle/input/**/{zname}", recursive=True)
    if zips:                                                   # 2a) zip trong /kaggle/input
        z, downloaded = zips[0], False
    elif drive:                                                # 2b) tai tu Drive
        z, downloaded = f"/kaggle/working/{zname}", True
        print(f"tai {zname} tu Google Drive ...", flush=True)
        drive_download(drive, z)
    else:
        print(f"!! THIEU {d}: Add Input mot Kaggle Dataset co {zname}, hoac dan link Drive vao MLM_CKPTS")
        continue
    tmp = f"/kaggle/working/_unzip/{tail.replace('/', '_')}"
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(z) as zf:
        print("  trong zip:", sorted({n.split('/')[0] for n in zf.namelist()}))
        zf.extractall(tmp)
    src = find_ckpt(tmp, tail)
    if not src:
        raise SystemExit(f"khong tim thay .../{tail}/config.json trong {z}")
    os.makedirs(os.path.dirname(d), exist_ok=True)
    shutil.move(src, d)
    shutil.rmtree(tmp, ignore_errors=True)
    if downloaded:
        os.remove(z)                                           # 0,9 GB khong can nua
    print(f"{d} <- {src.replace(tmp, '<zip>')}")

# kiem tra: config + tokenizer doc duoc, vocab khop bang embedding
from transformers import AutoTokenizer
for _, _, m in TRANSFORMERS:
    if not m:
        continue
    if not os.path.isfile(f"{m}/config.json"):
        raise SystemExit(f"chua co {m} -- xem cell tren")
    v = json.load(open(f"{m}/config.json"))["vocab_size"]
    tok = AutoTokenizer.from_pretrained(m)
    print(f"OK {m}: vocab {v:,} | tokenizer {len(tok):,} | 'maklu' -> {tok.tokenize('maklu')}")
print("du checkpoint")

## 3) Kiểm tra dữ liệu

Chạy đúng hàm mà `train.py` dùng, để thấy trước bao nhiêu câu val được thêm vào và nhãn ra sao.

In [ ]:
import pandas as pd
from src.utils.config import load_config
from src.data.preprocessing import ensure_processed, add_val_leak
from src.data.dataset import load_split
cfg = load_config("configs/base.yaml", ["data.val_leak_labels=true"], task="a")
ensure_processed(cfg)
tr, va, te = load_split(cfg, "train"), load_split(cfg, "val"), load_split(cfg, "test")
aug = add_val_leak(cfg, tr, va) if VAL_LEAK else tr.assign(source="train")
print(f"\ntrain goc: fit {int((tr.is_val == 0).sum())} + held-out {int((tr.is_val == 1).sum())}")
print(f"sau khi them val: fit {int((aug.is_val == 0).sum())} + held-out {int((aug.is_val == 1).sum())}  (held-out phai giu nguyen)")
print("nguon cua phan fit:", aug[aug.is_val == 0].source.value_counts().to_dict())
print("test:", len(te), "cau -- chi de du doan")

## 4) Kế hoạch

In danh sách run (đúng tên `train.py` sẽ tạo) trước khi train. Run đã xong sẽ tự bỏ qua.

In [ ]:
from src.utils.config import run_name
fmt = lambda v: str(v).replace(" ", "")
VL = ["data.val_leak_labels=true"] if VAL_LEAK else []

def name_of(conf, sets, seed=None, suffix=None):
    return run_name(load_config(f"configs/{conf}.yaml", sets, task="a", seed=seed, run_suffix=suffix))

JOBS = []      # (nhom, ten model, config, sets, seed, suffix, run name)
for c in TFIDF:
    sets = [f"model.clf={c}"] + VL
    JOBS.append(("tfidf", c, "tfidf", sets, None, None, name_of("tfidf", sets)))
for label, conf, mname in TRANSFORMERS:
    for s in SEEDS:
        sets = [f"training.epochs={EPOCHS}", f"training.early_stopping_patience={EPOCHS}"] + VL \
               + ([f"model.name={mname}"] if mname else [])
        JOBS.append(("trans", label, conf, sets, s, SUFFIX, name_of(conf, sets, s, SUFFIX)))
for g, label, conf, sets, s, suf, n in JOBS:
    print(f"  {g:6s} {label:7s} seed {s if s else '-':>3}  ->  {n}")
print(f"\n{len(JOBS)} run (ban 90%)" + (" + ban 100% sau do" if RUN_FULL else ""))

## 4b) Kiểm tra kết quả đã có (tránh train lại)

Nếu bạn *Add Input* file zip kết quả của lần chạy trước (vd `results_final_noleak.zip`), cell
dưới **khôi phục** các run trong đó vào `results/a/`. `train.py` tự **bỏ qua** run đã xong (cùng
cấu hình), nên các cell train phía sau **chỉ train phần còn thiếu**:
- đủ hết → chỉ train LLM (nếu `USE_LLM`);
- thiếu run nào → train bù run đó trước, rồi mới train LLM.

In [ ]:
import os, glob, re, zipfile
if RESULTS_ZIP.startswith("http"):                         # link Google Drive -> tai ve
    import gdown
    local = "/kaggle/working/results_prev.zip"
    if not os.path.isfile(local):
        m = re.search(r"/d/([\w-]+)", RESULTS_ZIP) or re.search(r"[?&]id=([\w-]+)", RESULTS_ZIP)
        gdown.download(id=m.group(1), output=local, quiet=False)
    if not zipfile.is_zipfile(local):
        raise SystemExit("file tai tu RESULTS_ZIP khong phai zip -- link da de 'Anyone with the link' chua?")
    zips = [local]
else:
    zips = [RESULTS_ZIP] if RESULTS_ZIP else sorted(glob.glob("/kaggle/input/**/results*.zip", recursive=True))
for zp in zips:                                             # zip phai cung che do val-leak voi notebook
    with zipfile.ZipFile(zp) as z:
        runs = {m.group(1) for x in z.namelist() for m in [re.match(r"(?:.*/)?results/a/([^/]+)/", x)] if m}
    has_vl = any("_vl" in r for r in runs)
    if runs and has_vl != VAL_LEAK:
        print(f"!! {os.path.basename(zp)} la ban {'CO' if has_vl else 'KHONG'} val-leak nhung VAL_LEAK={VAL_LEAK}: "
              f"ten run se khong khop -> moi run se bi TRAIN LAI. Doi VAL_LEAK={has_vl} de dung lai ket qua.")
restored = 0
for zp in zips:
    with zipfile.ZipFile(zp) as z:
        for member in z.namelist():
            m = re.match(r"(?:.*/)?results/a/([^/]+)/([^/]+)$", member)
            if not m or os.path.exists(f"results/a/{m.group(1)}/{m.group(2)}"):
                continue                                   # khong de len ket qua dang co
            os.makedirs(f"results/a/{m.group(1)}", exist_ok=True)
            with open(f"results/a/{m.group(1)}/{m.group(2)}", "wb") as f:
                f.write(z.read(member))
            restored += 1
print(f"khoi phuc {restored} file tu {len(zips)} zip: {[os.path.basename(z) for z in zips]}")

def is_done(name, scored=True):
    d = f"results/a/{name}"
    return os.path.isfile(f"{d}/test.npy") and os.path.isfile(f"{d}/config.yaml") and \
           (not scored or os.path.isfile(f"{d}/eval.npy"))
have = [n for *_, n in JOBS if is_done(n)]
miss = [n for *_, n in JOBS if not is_done(n)]
print(f"ban 90%: da co {len(have)}/{len(JOBS)} run" + (f" | THIEU: {miss}" if miss else " -> khong can train lai"))
full_have = sorted(os.path.basename(d) for d in glob.glob("results/a/*_full*") if is_done(os.path.basename(d), False))
print(f"ban 100% da co: {len(full_have)} run")
print("=> " + ("chi train LLM" if USE_LLM and not miss else
               "train bu run thieu" + (" roi train LLM" if USE_LLM else "")))

## 5) Train bản 90% (có điểm held-out)

TF-IDF chạy trên CPU trong vài giây. Mỗi run transformer khoảng 12–13 phút (Task A, thêm 13% dữ
liệu), 9 run tổng cộng khoảng 2 giờ. Mỗi run tự sinh file nộp riêng cho val và test.

In [ ]:
import time
t0 = time.time()
for n, (g, label, conf, sets, s, suf, name) in enumerate(JOBS, 1):
    args = " ".join(sets)
    extra = (f" --seed {s}" if s else "") + (f" --run_suffix {suf}" if suf else "")
    print("=" * 72, f"\n[{n}/{len(JOBS)}] {name} | +{(time.time() - t0) / 60:.1f} phut", flush=True)
    !python train.py --config configs/{conf}.yaml --task a --set {args}{extra}
print(f"\nxong trong {(time.time() - t0) / 60:.1f} phut")

## 6) Train bản 100% (`_full`)

Gộp cả 639 câu held-out vào phần fit. Không còn gì để chấm điểm, nên mỗi run dùng lại lựa chọn
của bản 90% tương ứng: **C/alpha** tốt nhất (TF-IDF) và **số epoch tốt nhất** (trung vị theo 3
seed của từng transformer). Lịch LR chạy đúng số epoch đó rồi giữ epoch cuối.

In [ ]:
import json, statistics
FULL_JOBS = []
if RUN_FULL:
    for g, label, conf, sets, s, suf, name in JOBS:
        m = json.load(open(f"results/a/{name}/metrics.json"))
        if g == "tfidf":
            fsets = sets + ["data.use_valdataset=false", f"model.param_grid=[{m['best_C']}]"]
        else:
            eps = [json.load(open(f"results/a/{n2}/metrics.json"))["best_epoch"]
                   for g2, l2, *_, n2 in JOBS if g2 == "trans" and l2 == label]
            ep = max(1, int(round(statistics.median(eps))))
            fsets = [x for x in sets if not x.startswith("training.")] + \
                    [f"training.epochs={ep}", f"training.early_stopping_patience={ep}", "data.use_valdataset=false"]
        FULL_JOBS.append((g, label, conf, fsets, s, suf, name_of(conf, fsets, s, suf)))
    t0 = time.time()
    for n, (g, label, conf, sets, s, suf, name) in enumerate(FULL_JOBS, 1):
        args = " ".join(sets)
        extra = (f" --seed {s}" if s else "") + (f" --run_suffix {suf}" if suf else "")
        print("=" * 72, f"\n[{n}/{len(FULL_JOBS)}] {name} | +{(time.time() - t0) / 60:.1f} phut", flush=True)
        !python train.py --config configs/{conf}.yaml --task a --set {args}{extra}
    print(f"\nxong trong {(time.time() - t0) / 60:.1f} phut")

Ensemble (tầng 1) = trung bình seed → trung bình model trong nhóm → `W_TRANS` × transformer +
(1 − `W_TRANS`) × TF-IDF. Định nghĩa ở đây vì tầng 2 cần nó để biết câu nào không chắc.

In [ ]:
import numpy as np
P = lambda name, split="eval": np.load(f"results/a/{name}/{split}.npy")
def model_probs(split, jobs, label):
    return np.mean([P(n, split) for g, l, *_, n in jobs if l == label], axis=0)
def group_probs(split, jobs, group, drop=()):
    labels = [l for l, *_ in TRANSFORMERS] if group == "trans" else TFIDF
    return np.mean([model_probs(split, jobs, l) for l in labels if l not in drop], axis=0)
def ensemble(split, jobs, w=W_TRANS, drop=()):
    return w * group_probs(split, jobs, "trans", drop) + (1 - w) * group_probs(split, jobs, "tfidf", drop)
for split in ("eval", "val", "test"):
    p1 = ensemble(split, JOBS)
    print(f"{split:4s}: {len(p1)} cau, {int((np.abs(p1[:, 1] - 0.5) < LLM_DELTA).sum())} cau khong chac (delta {LLM_DELTA})")

## 6b) Tầng 2 — LLM (nếu `USE_LLM`)

Mọi tham số nằm trong khối `LLM_*` của bảng điều khiển. Cell dưới ghi chúng ra
`configs/llm_notebook.yaml` (kế thừa `configs/llm_qwen7b.yaml`) rồi train bằng file đó.

| Tham số | Mặc định | Ý nghĩa | Khi nào đổi |
|---|---|---|---|
| `LLM_PRESET` | `qwen7b` | chọn sẵn model + 4-bit + batch cho 1 T4: `qwen7b`, `qwen3b`, `qwen1.5b`, `qwen0.5b`, `sarvam2b` | **không train nổi 7B** (OOM, hết thời gian) → `qwen3b`, rồi `qwen1.5b` |
| `LLM_MODEL` | theo preset | LLM làm encoder | dán tên model khác (Llama-3.1-8B, Gemma-2-9B cần HF token) |
| `LLM_4BIT` | True | nạp trọng số 4-bit | bắt buộc với 7B trên T4 |
| `LLM_LORA_R` / `_ALPHA` / `_DROP` | 16 / 32 / 0,05 | kích thước LoRA (chỉ phần này + head được train) | r=32 nếu LLM học chưa đủ; r=8 nếu overfit |
| `LLM_EPOCHS` | 3 | số epoch, cũng là độ dài lịch LR | 7B overfit sau epoch 3 (train F1 → 1,0, eval F1 giảm): đừng tăng lên 10 |
| `LLM_PATIENCE` | 1 | dừng sớm sau bấy nhiêu epoch eval F1 không tăng | = `LLM_EPOCHS` để tắt dừng sớm |
| `LLM_LR` | 1e-4 | learning rate | 5e-5 nếu loss dao động |
| `LLM_BATCH` × `LLM_GRAD_ACCUM` | 8 × 4 | batch hiệu dụng 32 | OOM → 4 × 8 |
| `LLM_MAX_LEN` | 192 | độ dài tối đa (token) | giảm nếu OOM |
| `LLM_GRAD_CKPT` | True | tiết kiệm bộ nhớ | tắt nếu dư VRAM (nhanh hơn ~30%) |
| `LLM_AUX_TARGET` / `LLM_TARGET_W` | True / 0,5 | head phụ "đối tượng bị tấn công" | 0 = tắt, để đo xem head phụ có giúp không |
| `LLM_PROMPT` | khung tiếng Anh | câu bọc mỗi bình luận | phải giữ `{text}` |

Tên run tự thêm hậu tố theo các giá trị khác mặc định, nên đổi tham số không đè lên lần train trước.
Đã có kết quả (từ zip) thì tự bỏ qua. Trên 1 T4, 7B mất khoảng 40 phút mỗi epoch, tức khoảng 2 giờ với 3 epoch.

**Không train (`LLM_MODE = 'zeroshot'` / `'fewshot'`)**: chạy `llm_prompt.py` thay vì `train.py`.
Chỉ dùng `LLM_MODEL`, `LLM_4BIT`, `LLM_K` và `VAL_LEAK` (các tham số train bị bỏ qua); model phải là bản
*-Instruct (có chat template, nên `sarvam2b` không dùng được).

- LLM không sinh câu trả lời: P(Hate) = softmax của logit token đầu của "Hate" và "Non-Hate".
- `fewshot`: mỗi câu kèm `LLM_K` câu **giống nhất** trong phần fit (TF-IDF n-gram ký tự), có nhãn.
  Ví dụ chỉ lấy từ phần fit, không bao giờ từ held-out, nên điểm held-out vẫn trung thực.
- LLM chưa train thường lệch hẳn về một nhãn. Vì vậy điểm được dời một hằng số sao cho tỉ lệ đoán Hate
  trên held-out bằng tỉ lệ Hate của phần fit (~49%). Bước này chỉ dùng tỉ lệ nhãn, không dùng nhãn
  held-out. `metrics.json` ghi cả F1 trước và sau hiệu chỉnh.
- 7B 4-bit trên 1 T4: zeroshot ~10–15 phút, fewshot k=8 ~30–40 phút cho 3 tập (639 + 806 + 806 câu).

`LLM_SCOPE` (chỉ cho zeroshot/fewshot) chọn câu nào đưa cho LLM:

| `LLM_SCOPE` | LLM chấm | Thời gian (7B, fewshot k=8) | Dùng được cho |
|---|---|---|---|
| `'band'` | chỉ câu ensemble **không chắc** (\|P − 0,5\| < `LLM_DELTA`), khoảng 1/4 số câu | ~10 phút | 2 tầng (`a_test_final_llm`) |
| `'all'` | **mọi** câu held-out/val/test | ~30–40 phút | 2 tầng + **LLM một mình** (`a_test_final_llmonly`) |

- Với `'band'`, câu ngoài band giữ xác suất của ensemble. Vì vậy hàng "LLM một mình" trong bảng mục 7
  chính là "2 tầng với `LLM_MIX` = 0".
- Đổi `LLM_DELTA` thì phải chấm lại: tên run có `_band<delta>`, nên lần chấm cũ không bị đè.
- Band gồm câu không chắc của bản 90% **hoặc** bản 100% (nếu `RUN_FULL`), nên cả hai file 2 tầng đều dùng được.

Đây là phép thử rẻ: nếu few-shot đã sửa được nhiều lỗi trong vùng không chắc (bảng ghép 2 tầng ở mục 7),
có thể không cần fine-tune.

In [ ]:
import yaml
LLM_NAME = None
if USE_LLM and LLM_MODE in ("zeroshot", "fewshot"):
    if LLM_4BIT:
        !pip -q install bitsandbytes accelerate
    assert LLM_SCOPE in ("band", "all"), f"LLM_SCOPE phai la band / all, khong phai {LLM_SCOPE!r}"
    from llm_prompt import default_run_name
    band_args = ""
    if LLM_SCOPE == "band":
        # cau khong chac cua ensemble 90% HOAC 100%; ngoai band LLM giu xac suat cua ensemble 90%
        z = {}
        for split in ("eval", "val", "test"):
            p1 = ensemble(split, JOBS)
            mask = np.abs(p1[:, 1] - 0.5) < LLM_DELTA
            if split != "eval" and FULL_JOBS:                      # ban 100% khong co eval
                mask |= np.abs(ensemble(split, FULL_JOBS)[:, 1] - 0.5) < LLM_DELTA
            z[f"{split}_p1"], z[f"{split}_mask"] = p1, mask
            print(f"  {split}: LLM cham {int(mask.sum())}/{len(mask)} cau")
        np.savez("results/llm_band.npz", **z)
        band_args = f"--band_file results/llm_band.npz --band_delta {LLM_DELTA}"
    LLM_NAME = default_run_name(LLM_MODEL, LLM_MODE, LLM_K, VAL_LEAK, LLM_DELTA if LLM_SCOPE == "band" else None)
    print("LLM run:", LLM_NAME, "(da co -> bo qua)" if is_done(LLM_NAME) else "(khong train)")
    if not is_done(LLM_NAME):
        t0 = time.time()
        !python llm_prompt.py --model {LLM_MODEL} --mode {LLM_MODE} --k {LLM_K} --seed {LLM_SEED} {"--load_in_4bit" if LLM_4BIT else ""} {"--val_leak" if VAL_LEAK else ""} {band_args}
        print(f"-> {(time.time() - t0) / 60:.1f} phut")
    if not is_done(LLM_NAME):
        raise SystemExit("LLM chua co ket qua -- xem log o tren (OOM? model khong co chat template?)")
elif USE_LLM:
    assert LLM_MODE == "finetune", f"LLM_MODE phai la finetune / zeroshot / fewshot, khong phai {LLM_MODE!r}"
    !pip -q install peft bitsandbytes accelerate
    # bang dieu khien -> file config (prompt nhieu dong co ':' va '"' nen khong truyen qua --set)
    llm_cfg = {
        "_base_": "llm_qwen7b.yaml",
        "model": {"name": LLM_MODEL, "load_in_4bit": bool(LLM_4BIT) or None, "aux_target": bool(LLM_AUX_TARGET) or None,
                  "lora": {"r": LLM_LORA_R, "alpha": LLM_LORA_ALPHA, "dropout": LLM_LORA_DROP}},
        "data": {"max_len": LLM_MAX_LEN, "prompt": LLM_PROMPT, "val_leak_labels": bool(VAL_LEAK) or None},
        "training": {"epochs": LLM_EPOCHS, "early_stopping_patience": LLM_PATIENCE, "lr": LLM_LR,
                     "head_lr": LLM_LR, "batch_size": LLM_BATCH, "grad_accum": LLM_GRAD_ACCUM,
                     "grad_checkpointing": bool(LLM_GRAD_CKPT), "target_weight": LLM_TARGET_W},
    }
    yaml.safe_dump(llm_cfg, open("configs/llm_notebook.yaml", "w", encoding="utf-8"),
                   allow_unicode=True, sort_keys=False)
    # hau to ten run: chi ghi gia tri khac mac dinh, de lan train khac tham so khong de len nhau
    base = yaml.safe_load(open("configs/llm_qwen7b.yaml", encoding="utf-8"))
    llm_suffix = ""
    for tag, val, dflt in (("m", LLM_MODEL.split("/")[-1].lower()[:14], "qwen2.5-7b-ins"),
                           ("e", LLM_EPOCHS, base["training"]["epochs"]), ("lr", LLM_LR, _p["lr"]),
                           ("bs", LLM_BATCH * LLM_GRAD_ACCUM, 32), ("q", int(LLM_4BIT), int(_p["four_bit"])),
                           ("r", LLM_LORA_R, 16), ("len", LLM_MAX_LEN, base["data"]["max_len"]),
                           ("tw", LLM_TARGET_W, base["training"]["target_weight"])):
        if val != dflt:
            llm_suffix += f"_{tag}{val:g}" if isinstance(val, (int, float)) else f"_{tag}{re.sub(r'[^a-z0-9.-]', '', val)}"
    llm_suffix = llm_suffix or None
    LLM_NAME = run_name(load_config("configs/llm_notebook.yaml", task="a", seed=LLM_SEED, run_suffix=llm_suffix))
    print("LLM run:", LLM_NAME, "(da co -> bo qua)" if is_done(LLM_NAME) else "")
    print(yaml.safe_dump(llm_cfg, allow_unicode=True, sort_keys=False))
    t0 = time.time()
    !python train.py --config configs/llm_notebook.yaml --task a --seed {LLM_SEED} {"--run_suffix " + llm_suffix if llm_suffix else ""}
    print(f"-> {(time.time() - t0) / 60:.1f} phut")
    if not is_done(LLM_NAME):
        raise SystemExit("LLM chua co ket qua -- xem log o tren (OOM? thieu peft/bitsandbytes?)")

## 7) Kết quả trên held-out (chỉ bản 90%)

Từng run, trung bình seed của từng model, từng nhóm, ensemble cuối, và các biến thể bỏ bớt một
model. Lát held-out có 639 câu, nên chênh lệch dưới khoảng 0,02 là trong mức nhiễu. **Đừng chọn
`W` theo bảng này**: đó chính là tối ưu trên held-out mà ta đang tránh.

In [ ]:
from sklearn.metrics import f1_score
y = tr.y.to_numpy()[tr.is_val.to_numpy() == 1]
f1 = lambda p: f1_score(y, p.argmax(1), average="macro")

rows = [{"run": n, "nhom": g, "model": l, "macro_f1": round(f1(P(n)), 4)} for g, l, *_, n in JOBS]
display(pd.DataFrame(rows))

summary = [{"to hop": f"{l} (TB {len(SEEDS)} seed)", "macro_f1": f1(model_probs("eval", JOBS, l))} for l, *_ in TRANSFORMERS]
summary += [{"to hop": "nhom transformer", "macro_f1": f1(group_probs("eval", JOBS, "trans"))},
            {"to hop": "nhom TF-IDF", "macro_f1": f1(group_probs("eval", JOBS, "tfidf"))},
            {"to hop": f"ENSEMBLE (W={W_TRANS})", "macro_f1": f1(ensemble("eval", JOBS))}]
summary += [{"to hop": f"ensemble bo {d}", "macro_f1": f1(ensemble("eval", JOBS, drop=(d,)))}
            for d in [l for l, *_ in TRANSFORMERS] + TFIDF]
summary += [{"to hop": f"(tham khao) W={w}", "macro_f1": f1(ensemble("eval", JOBS, w))} for w in (0.4, 0.5, 0.7, 0.8)]
display(pd.DataFrame(summary).round(4))

**Ghép 2 tầng** (chỉ khi `USE_LLM`). Câu nào ensemble **chắc chắn** (|P(Hate) − 0,5| ≥ `LLM_DELTA`)
giữ nguyên dự đoán của ensemble; câu nào **không chắc** thì trộn với LLM theo `LLM_MIX`.

Mỗi delta có hai cột F1:
- **trộn**: theo `LLM_MIX` đang đặt, P = `LLM_MIX` × P_ensemble + (1 − `LLM_MIX`) × P_LLM;
- **LLM quyết hoàn toàn** (mix 0): trong vùng, nhãn lấy hẳn theo LLM.

Bảng quan trọng nhất là **độ chính xác TRONG vùng không chắc**: ensemble và LLM, trên cùng các
câu. LLM chỉ có ích nếu nó đúng nhiều hơn ensemble ở đúng vùng đó.

In [ ]:
def cascade(split, jobs, delta=LLM_DELTA, mix=LLM_MIX):
    p1 = ensemble(split, jobs)
    p2 = P(LLM_NAME, split)
    band = np.abs(p1[:, 1] - 0.5) < delta
    p = p1.copy()
    p[band] = mix * p1[band] + (1 - mix) * p2[band]
    return p, band

LLM_BAND_ONLY = USE_LLM and LLM_MODE != "finetune" and LLM_SCOPE == "band"
if USE_LLM:
    p1, p2 = ensemble("eval", JOBS), P(LLM_NAME)
    acc = lambda p, m: (p[m].argmax(1) == y[m]).mean() if m.any() else float("nan")
    rows = [{"to hop": "ENSEMBLE (tang 1)", "macro_f1": f1(p1)},
            {"to hop": "LLM mot minh" + (" (= 2 tang, mix 0: LLM chi cham trong band)" if LLM_BAND_ONLY else ""), "macro_f1": f1(p2)},
            {"to hop": f"ensemble + LLM nhu thanh vien thu 7 (TB 0,5/0,5)", "macro_f1": f1(0.5 * p1 + 0.5 * p2)}]
    for d in sorted({0.1, 0.2, 0.3, 0.5, LLM_DELTA}):
        if LLM_BAND_ONLY and d > LLM_DELTA:                        # ngoai band LLM chua cham
            continue
        pc, band = cascade("eval", JOBS, delta=d)
        rows.append({"to hop": f"CASCADE delta={d}" + ("  <- dang dung" if d == LLM_DELTA else ""),
                     "macro_f1": f1(pc), "F1 LLM quyet hoan toan (mix 0)": f1(cascade("eval", JOBS, delta=d, mix=0)[0]),
                     "cau vao vung khong chac": int(band.sum()),
                     "acc ensemble trong vung": round(acc(p1, band), 3), "acc LLM trong vung": round(acc(p2, band), 3)})
    display(pd.DataFrame(rows).round(4).rename(columns={"macro_f1": f"macro_f1 (tron, mix {LLM_MIX:g})"}))
    print("Luu y: chon delta theo bang nay la toi uu tren held-out (639 cau) -> chi nen doi neu chenh ro.")

## 8) File nộp

Ensemble với cùng trọng số cho:
- **bản 90%** → `a_test_final` (và `a_val_final`)
- **bản 100%** → `a_test_final_full` (và `a_val_final_full`)
- nếu `USE_LLM`: **2 tầng** → `a_test_final_llm` và `a_test_final_full_llm` (trộn theo `LLM_MIX`)
- nếu `USE_LLM`: **2 tầng, LLM quyết hoàn toàn trong vùng** → `a_test_final_llm_mix0` và `a_test_final_full_llm_mix0`
  (bỏ qua nếu `LLM_MIX` = 0, vì khi đó `final_llm` đã là bản này)
- nếu `USE_LLM` và LLM đã chấm **mọi** câu (finetune, hoặc `LLM_SCOPE = 'all'`): **LLM một mình** → `a_test_final_llmonly`

**Nộp file `test`.** File `val` chỉ sinh ra cho đủ bộ: model đã học chính các câu đó.

In [ ]:
from src.evaluation.submission import write_submission
from src.data.dataset import label_names
labels = label_names("a")
outs = {}
for tag, jobs in [("final", JOBS)] + ([("final_full", FULL_JOBS)] if RUN_FULL else []):
    for split, df in (("val", va), ("test", te)):
        p = ensemble(split, jobs)
        outs[(tag, split)] = p
        write_submission(df.id.values, p, labels, f"results/submissions/a_{split}_{tag}")
if USE_LLM:
    # 2 tang: ensemble (90% hoac 100%) + LLM (ban 90%) trong vung khong chac
    # tag -> (jobs, mix): ban tron theo LLM_MIX, va ban LLM quyet hoan toan trong vung (mix 0)
    llm_tags = {"final_llm": (JOBS, LLM_MIX)}
    if RUN_FULL:
        llm_tags["final_full_llm"] = (FULL_JOBS, LLM_MIX)
    if LLM_MIX != 0:
        llm_tags["final_llm_mix0"] = (JOBS, 0)
        if RUN_FULL:
            llm_tags["final_full_llm_mix0"] = (FULL_JOBS, 0)
    for tag, (jobs, mix) in llm_tags.items():
        for split, df in (("val", va), ("test", te)):
            p, band = cascade(split, jobs, mix=mix)
            outs[(tag, split)] = p
            write_submission(df.id.values, p, labels, f"results/submissions/a_{split}_{tag}")
            if split == "test":
                changed = int((p.argmax(1) != ensemble(split, jobs).argmax(1)).sum())
                print(f"  {tag} (mix {mix:g}): {int(band.sum())} cau test vao vung khong chac, LLM doi {changed} nhan")
    if not LLM_BAND_ONLY:
        # LLM tu quyet ca file (khong dung ensemble)
        for split, df in (("val", va), ("test", te)):
            p = P(LLM_NAME, split)
            outs[("final_llmonly", split)] = p
            write_submission(df.id.values, p, labels, f"results/submissions/a_{split}_final_llmonly")
        changed = int((outs[("final_llmonly", "test")].argmax(1) != outs[("final", "test")].argmax(1)).sum())
        print(f"  final_llmonly: LLM mot minh khac ensemble o {changed}/{len(te)} cau test")
if RUN_FULL:
    a, b = outs[("final", "test")].argmax(1), outs[("final_full", "test")].argmax(1)
    print(f"\nban 90% va ban 100% dong y tren {(a == b).mean():.1%} cau test")

## 9) Tải về

In [ ]:
%cd /kaggle/working/repo
!mkdir -p /kaggle/working/final_submissions
!cp -r results/submissions/a_test_final* results/submissions/a_val_final* /kaggle/working/final_submissions/
!ls -R /kaggle/working/final_submissions | head -30
!zip -r -q /kaggle/working/results_final.zip results logs
!ls -lh /kaggle/working/results_final.zip